# 03 — Gold Layer: Analytical Aggregations

This notebook demonstrates the **Gold layer** — the consumption-ready analytical tables.

**Tables produced:**
| Table | Grain | Description |
|---|---|---|
| `daily_metrics` | ticker × trade_date | OHLCV + rolling volatility, avg volume, cumulative return |
| `portfolio_summary` | ticker | Period totals: total return, avg volume, avg volatility |
| `monthly_returns` | ticker × year × month | Monthly OHLC and return |

In [ ]:
import sys
sys.path.insert(0, "..")

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
pl.Config.set_tbl_rows(20)

## 1. Build Gold tables

In [ ]:
from pipelines.gold_pipeline import GoldPipeline

tables = GoldPipeline().run()

for name, df in tables.items():
    print(f"{name:25s}: {len(df):>6,} rows  |  cols: {df.columns}")

## 2. Daily metrics overview

In [ ]:
daily = tables.get("daily_metrics")
if daily is not None:
    daily.head(10)

In [ ]:
# Cumulative returns comparison
if daily is not None:
    top5 = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]
    plot_df = daily.filter(pl.col("ticker").is_in(top5)).sort("trade_date").to_pandas()

    fig = px.line(
        plot_df,
        x="trade_date",
        y="cum_return",
        color="ticker",
        title="Cumulative Returns — Gold Layer",
        labels={"trade_date": "Date", "cum_return": "Cumulative Return"},
    )
    fig.update_yaxes(tickformat=".1%")
    fig.show()

In [ ]:
# Rolling 20-day volatility
if daily is not None:
    fig2 = px.line(
        plot_df,
        x="trade_date",
        y="volatility_20d",
        color="ticker",
        title="20-Day Annualised Volatility",
        labels={"trade_date": "Date", "volatility_20d": "Volatility (annualised)"},
    )
    fig2.update_yaxes(tickformat=".1%")
    fig2.show()

## 3. Portfolio summary

In [ ]:
summary = tables.get("portfolio_summary")
if summary is not None:
    print("Top performers:")
    display(summary.head(10).to_pandas())

In [ ]:
# Return vs Volatility scatter
if summary is not None:
    sum_pd = summary.to_pandas()
    fig3 = px.scatter(
        sum_pd.dropna(subset=["total_return", "avg_volatility"]),
        x="avg_volatility",
        y="total_return",
        text="ticker",
        title="Risk vs Return (period)",
        labels={"avg_volatility": "Avg Volatility", "total_return": "Total Return"},
    )
    fig3.update_traces(textposition="top center")
    fig3.update_xaxes(tickformat=".1%")
    fig3.update_yaxes(tickformat=".1%")
    fig3.show()

## 4. Monthly returns heatmap

In [ ]:
monthly = tables.get("monthly_returns")
if monthly is not None:
    monthly.head(10)

In [ ]:
# Heatmap: ticker vs month → monthly return
import seaborn as sns
import matplotlib.pyplot as plt

if monthly is not None:
    pivot = (
        monthly
        .filter(pl.col("ticker").is_in(top5))
        .with_columns(pl.concat_str([pl.col("year").cast(pl.Utf8), pl.lit("-"), pl.col("month").cast(pl.Utf8)]).alias("ym"))
        .pivot(values="month_return", index="ticker", on="ym")
        .to_pandas()
        .set_index("ticker")
        .astype(float)
    )

    fig4, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1%",
        cmap="RdYlGn",
        center=0,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title("Monthly Returns Heatmap")
    plt.tight_layout()
    plt.show()

## 5. Read persisted Gold tables

In [ ]:
from processing.gold.aggregate import read_gold

gold_daily = read_gold("daily_metrics")
print(f"Persisted daily_metrics rows: {len(gold_daily):,}")